# 2. Build the training data

Writes the training set from the units only: question and answer pairs in several wordings, each answer checked against its unit.

Settings: Accelerator = GPU T4 x2, Internet = On. Use **Save Version > Save & Run All (Commit)** so it keeps running with the browser closed. Its output (`train.jsonl`) is the input of notebook 3.

In [ ]:
ROUNDS = 2    # question-writing passes per unit (about 8 wordings each)
EXTRA = True  # run 2: one more pass of indirect questions, and near-miss refusals (run 1: False)

In [ ]:
REPO_URL = "https://github.com/Naveenasri02/qwen25-lora-book-pilot.git"

import glob, json, os, shutil, subprocess, sys, zipfile
os.environ["HF_HOME"] = "/tmp/hf"          # big downloads go to scratch space, not the 20 GB output folder
REPO = "/tmp/repo"
if not os.path.exists(REPO):
    # the project comes from an attached dataset if there is one, otherwise from GitHub
    attached = glob.glob("/kaggle/input/**/pilot/config.py", recursive=True)
    zipped = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if attached:
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), REPO)
    elif zipped:
        zipfile.ZipFile(zipped[0]).extractall(REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO], check=True)
sys.path.insert(0, REPO)
from pilot import judge, ollama_client as oc
from pilot.config import (BASE_OLLAMA, FRESH_QUESTIONS_PATH, JUDGE_OLLAMA, QUESTIONS_PATH, SEED,
                          TUNED_OLLAMA, UNITS_PATH)
OUT = "/kaggle/working"
questions = json.loads(QUESTIONS_PATH.read_text(encoding="utf-8"))
fresh = json.loads(FRESH_QUESTIONS_PATH.read_text(encoding="utf-8"))   # second exam, written for run 2
print(len(questions), "exam questions and", len(fresh), "fresh exam questions loaded")

In [ ]:
from pilot import traindata
from pilot.config import PRACTICAL_QUESTIONS_PATH, REFUSAL_QUESTIONS_PATH
oc.start_server()
oc.pull(BASE_OLLAMA)
units = json.loads(UNITS_PATH.read_text(encoding="utf-8"))
saved = f"{REPO}/data/generated/gen_cache.jsonl"   # replies kept from an earlier, interrupted run
if os.path.exists(saved) and not os.path.exists(f"{OUT}/gen_cache.jsonl"):
    shutil.copy(saved, f"{OUT}/gen_cache.jsonl")
examples = traindata.build(oc.chat_many, BASE_OLLAMA, units, questions + fresh,   # neither exam may leak
                           traindata.read_lines(REFUSAL_QUESTIONS_PATH),
                           traindata.read_lines(PRACTICAL_QUESTIONS_PATH),
                           rounds=ROUNDS, seed=SEED, cache_path=f"{OUT}/gen_cache.jsonl", extra=EXTRA)
traindata.save_jsonl(examples, f"{OUT}/train.jsonl")
for e in examples[:5]:
    print(e["kind"], "|", e["messages"][1]["content"], "->", e["messages"][2]["content"][:160])